# ResearchGPT
RAG pipeline for question answering over research papers with citations.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pymupdf groq

In [ ]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Groq API key: ')

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_paths = list(uploaded.keys())

In [ ]:
import fitz
from dataclasses import dataclass

@dataclass
class PageContent:
    doc_name: str
    page_number: int
    text: str

def extract_pages(pdf_path):
    doc_name = pdf_path.split('/')[-1]
    pages = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc):
            text = page.get_text('text').strip()
            if text:
                pages.append(PageContent(doc_name, i + 1, text))
    return pages

all_pages = []
for p in pdf_paths:
    all_pages.extend(extract_pages(p))

len(all_pages)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)

documents = []
for page in all_pages:
    for piece in splitter.split_text(page.text):
        documents.append(Document(
            page_content=piece,
            metadata={'doc_name': page.doc_name, 'page_number': page.page_number}
        ))

len(documents)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={'normalize_embeddings': True}
)

vector_store = FAISS.from_documents(documents, embeddings)
vector_store.index.ntotal

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ['GROQ_API_KEY'])

PROMPT_TEMPLATE = '''You are ResearchGPT, an assistant that answers questions strictly using the provided research paper excerpts.
Only use information found in the CONTEXT. If the answer isn't there, say so honestly. Be concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:'''

def ask(question, k=4):
    results = vector_store.similarity_search_with_relevance_scores(question, k=k)
    context = '\n\n'.join(
        f"[Source {i+1} | {doc.metadata['doc_name']} | Page {doc.metadata['page_number']}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
        max_tokens=800,
    )
    answer = response.choices[0].message.content.strip()

    print(answer)
    print()
    for doc, score in results:
        print(f"{doc.metadata['doc_name']} (page {doc.metadata['page_number']}, relevance {score:.3f})")
    return answer, results

In [ ]:
_ = ask('What problem does this paper try to solve, and what method do they propose?')